## Analyse the MBJ band gap calculation results and export

In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [9]:
from aiida import orm
from monty.serialization import loadfn, dumpfn
import numpy as np
from aiida_vasp.workchains import VaspHybridBandsWorkChain
from aiida_vasp.workchains.v2 import VaspHybridBandUpdater, VaspRelaxUpdater
from aiida_grouppathx import GroupPathX
from tqdm import tqdm
from aiida_grouppathx import GroupPathX

from aiida_user_addons.tools.sumo import get_sumo_bands_plotter, get_pmg_bandstructure

import pandas as pd

from tqdm import tqdm

In [8]:
from aiida_vasp.utils.compare_bands import get_band_properties_from_data

In [24]:
basepath = GroupPathX('hc-ternary-mbj')
#basepath = GroupPathX('hc-binary-mbj')
workpath = basepath['bandstructure_works']
nodes = [path.get_node() for path in workpath]
finished_ok = list(filter(lambda x: x.is_finished_ok, nodes)) 

In [25]:
len(finished_ok)

927

In [26]:
def show_bands(worknode):
    plotter = get_sumo_bands_plotter(worknode.outputs.band_structure, structure=worknode.outputs.primitive_structure)
    return plotter.get_plot()

In [27]:
records = []
for node in tqdm(finished_ok):
    bs = get_pmg_bandstructure(node.outputs.band_structure, node.outputs.primitive_structure)
    band_info = bs.get_band_gap()  # Using pymatgen band gap information which is more robust than that shipped in AiiDA vasp
    band_info['band_gap'] = band_info.pop('energy')
    out_dict =     {
        'structure': node.inputs.structure.get_pymatgen(),
        'formula': node.inputs.structure.get_formula(),
        'material_id': node.label.split()[1],
    }
    out_dict.update(band_info)
    records.append(
        out_dict
)
df = pd.DataFrame.from_records(records)   

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 927/927 [00:38<00:00, 24.34it/s]


In [28]:
df[df.direct == True].sort_values('band_gap')

,structure,formula,material_id,direct,transition,band_gap
179,"[[2.42296559 0. 2.42296559] Ca, [0. ...",Ca3OPb,mp-20273,True,\Gamma-\Gamma,0.2708
207,"[[0. 0. 0.] Eu, [ 2.40665171 -1.38948405 4.95...",Bi2EuMg2,mp-1006146,True,\Gamma-\Gamma,0.2730
771,"[[1.8363439 4.40376943 9.39275589] Cu, [ 5.62...",Cu8Ge4S12,mp-1225871,True,\Gamma-\Gamma,0.2872
53,[[-5.11124883e-07 -2.21324154e-07 -1.82053234e...,As4In4Sr2,mp-1205860,True,\Gamma-\Gamma,0.2927
567,"[[0. 0. 0.] Ce, [0.472153 4.87952687 1.33377...",CeP12Ru4,mp-10069,True,\Gamma-\Gamma,0.2932
...,...,...,...,...,...,...
18,"[[ 1.46295719 2.53391594 -2.0689344 ] Na, [-1...",Na3S4Sb,mp-10167,True,H-H,2.9881
238,"[[3.19391328 0. 1.53261599] Ga, [0. 0....",GaO4Sb,mp-1224786,True,\Gamma-\Gamma,3.0389
596,"[[0. 0. 3.01135145] Ga, [2.278...",Ga4O12Te2,mp-28931,True,\Gamma-\Gamma,3.2797
524,"[[0.86681655 0.61289479 1.48720256] Mg, [0.883...",Fe4Mg2O8,mp-608016,True,\Gamma-\Gamma,3.3033


In [29]:
from ase.visualize import view

Export the data to a csv

In [30]:
#df.pop('structure', None)

df = df.sort_values('band_gap')
df.drop(['structure'], axis=1).to_csv('mbj_gaps_tenary_2025_0425_pmg_info.csv')